# Workflow Patterns in Pure Python

**What you will build:** the four classic workflow patterns — chaining, routing, parallelization, and orchestrator-workers — each in a few lines of plain Python. The lesson underneath: for most tasks you **don't need an agent, or a framework**. You saw the model drive in 0.3; here *you* drive.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/04_workflow_patterns.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai pydantic

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    """One-shot helper: system + user in, text out."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready. Model:", MODEL)

## Agent vs workflow, in one line of code each

In 0.3 the model chose the steps. In a workflow, *you* write the steps as ordinary control flow. These patterns come from Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents); this is the same material as chapters 04a–04c of the n8n course, but here a "node" is just a function call.

```{note}
Rule of thumb from 0.0: reach for the simplest thing that works. If you can draw the flowchart in advance, it's a workflow — write the `if`s yourself. Save agents for when the path can't be known ahead of time.
```

## Pattern 1 — Prompt chaining

Break one hard task into a sequence where each step's output feeds the next. Each step is simpler, so each is more reliable. Here: a raw recipe becomes a beginner version, then a shopping list.

In [ ]:
recipe = "Beef lasagna: brown 500g beef, layer with tomato sauce, bechamel and pasta, bake 40 min."

simplified = ask("You simplify recipes for absolute beginners. Number every step.", recipe)
shopping   = ask("You are a shopping assistant. Output ONLY a grouped shopping list.", simplified)

print("SIMPLIFIED:\n", simplified[:300], "...\n")
print("SHOPPING LIST:\n", shopping)

That's the whole pattern: `output_1 -> input_2`. No framework, no agent — just two calls in a row.

## Pattern 2 — Routing

Classify the input first, then send it down a different path. This is where structured output (0.2) pays off: the classifier returns a **typed value**, and plain Python branches on it. (The fragile alternative — asking for free text and fishing with `if "billing" in reply` — is exactly the antipattern 0.2 taught you to avoid: one day the model answers "This is clearly a billing-related issue" and your substring match silently routes it wrong, or matches two categories at once.)

In [ ]:
from pydantic import BaseModel
from typing import Literal

class Route(BaseModel):
    category: Literal["billing", "technical", "account"]

def classify(ticket):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content":
             'Classify the support ticket. Reply with JSON: {"category": "billing"|"technical"|"account"}'},
            {"role": "user", "content": ticket}])
    return Route.model_validate_json(r.choices[0].message.content).category

def route_ticket(ticket):
    category = classify(ticket)     # a typed Literal, not a string to fish in

    # YOU control the branch — plain Python:
    if category == "billing":
        return ask("You are a billing specialist. Be precise about charges and refunds.", ticket)
    elif category == "technical":
        return ask("You are a technical support engineer. Give clear troubleshooting steps.", ticket)
    else:  # "account" — the Literal guarantees there is no fourth value
        return ask("You are an account specialist. Help with login and settings.", ticket)

print(route_ticket("I was double-charged for my plan this month."))

## Pattern 3 — Parallelization

When subtasks are independent, run them at the same time instead of one after another. With `AsyncOpenAI` and `asyncio.gather`, three calls take about as long as one. (Notebooks support top-level `await` directly.)

In [ ]:
import asyncio
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

async def analyze(aspect, text):
    r = await aclient.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role":"system","content":f"In one sentence, analyze the {aspect} of the review."},
                  {"role":"user","content":text}])
    return aspect, r.choices[0].message.content

review = "The laptop is fast and beautiful, but it runs hot and the battery barely lasts three hours."

# Fire all three at once and wait for all of them:
results = await asyncio.gather(
    analyze("sentiment", review),
    analyze("hardware quality", review),
    analyze("battery", review),
)
for aspect, out in results:
    print(f"{aspect:16}: {out}")

## Pattern 4 — Orchestrator-workers

A coordinator LLM breaks the task into subtasks, workers handle each — **in parallel**, because the subtasks are independent (pattern 3 nested inside pattern 4; the patterns compose) — and a final step combines them. This is the workflow that looks most like an agent, but *you* still fix the shape: plan, then work, then synthesize.

In [ ]:
from pydantic import BaseModel

class Plan(BaseModel):
    subtopics: list[str]

async def ask_async(system, user):
    r = await aclient.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

async def orchestrate(topic):
    # 1. Orchestrator: plan the subtopics (structured output from 0.2).
    #    Bare-bones on purpose — see 0.2's extract() for the retry-on-validation version.
    raw = await aclient.chat.completions.create(
        model=MODEL, temperature=0, response_format={"type":"json_object"},
        messages=[{"role":"system","content":"Break the topic into 3 subtopics. JSON: {\"subtopics\": [...]}"},
                  {"role":"user","content":topic}])
    plan = Plan.model_validate_json(raw.choices[0].message.content)

    # 2. Workers: independent subtasks -> run them all at once (pattern 3):
    parts = await asyncio.gather(*[
        ask_async("Write one concise paragraph.", f"Topic: {topic}\nSubtopic: {s}")
        for s in plan.subtopics
    ])

    # 3. Synthesize — needs ALL the parts, so this step stays sequential:
    return await ask_async("Combine these paragraphs into a coherent short article.", "\n\n".join(parts))

print(await orchestrate("Why bees matter to agriculture"))

## The thesis, measured

This notebook's opening claim — *for most tasks you don't need an agent* — shouldn't stay an opinion. Let's measure it. Same task (recipe → beginner version → shopping list), two implementations:

- **Workflow:** you call the two steps in order. Two model calls, by construction.
- **Agent:** the model gets both steps as *tools* and decides the order itself, with the 0.3 loop.

Both produce a shopping list. Watch what the agent's freedom costs:

In [ ]:
import time

USAGE = {"calls": 0, "tokens": 0}

def ask_counted(system, user):
    r = client.chat.completions.create(
        model=MODEL, temperature=0,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    USAGE["calls"] += 1
    USAGE["tokens"] += r.usage.total_tokens
    return r.choices[0].message.content

# ---------- Version A: the workflow (you drive) ----------
USAGE = {"calls": 0, "tokens": 0}; t0 = time.time()
s = ask_counted("You simplify recipes for absolute beginners. Number every step.", recipe)
_ = ask_counted("You are a shopping assistant. Output ONLY a grouped shopping list.", s)
workflow_stats = (USAGE["calls"], USAGE["tokens"], time.time() - t0)

# ---------- Version B: the agent (the model drives) ----------
# The same two steps, wrapped as tools. Note each tool is itself an LLM call.
def simplify_recipe(text: str) -> str:
    return ask_counted("You simplify recipes for absolute beginners. Number every step.", text)

def shopping_list(text: str) -> str:
    return ask_counted("You are a shopping assistant. Output ONLY a grouped shopping list.", text)

step_tools = [
    {"type": "function", "function": {
        "name": "simplify_recipe", "description": "Rewrite a recipe for absolute beginners.",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
    {"type": "function", "function": {
        "name": "shopping_list", "description": "Turn a (simplified) recipe into a grouped shopping list.",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}}},
]
STEP_FUNCTIONS = {"simplify_recipe": simplify_recipe, "shopping_list": shopping_list}

USAGE = {"calls": 0, "tokens": 0}; t0 = time.time()
messages = [{"role": "system", "content": "Use the tools to produce a beginner-friendly shopping list."},
            {"role": "user", "content": recipe}]
for _ in range(6):                                     # the 0.3 loop, compact
    r = client.chat.completions.create(model=MODEL, messages=messages, tools=step_tools, temperature=0)
    USAGE["calls"] += 1; USAGE["tokens"] += r.usage.total_tokens
    msg = r.choices[0].message
    messages.append(msg)
    if not msg.tool_calls:
        break
    for tc in msg.tool_calls:
        args = tc.function.arguments
        try:
            args = json.loads(args)
            result = STEP_FUNCTIONS[tc.function.name](**args)
        except Exception as e:
            result = f"Error: {e}"
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
agent_stats = (USAGE["calls"], USAGE["tokens"], time.time() - t0)

print(f"workflow: {workflow_stats[0]} calls, {workflow_stats[1]:>5} tokens, {workflow_stats[2]:.1f}s")
print(f"agent:    {agent_stats[0]} calls, {agent_stats[1]:>5} tokens, {agent_stats[2]:.1f}s")

Typical result: the agent needs **2–3× the calls and tokens** for the *same output* — every coordination turn re-sends the growing history, and each tool execution here is an LLM call on top. That's the price of letting the model drive. It's worth paying when the path genuinely can't be fixed in advance (0.3's weather-and-math question, open-ended research, debugging); it's waste when the flowchart was known all along, like here.

This is the criterion you'll use for real systems, so say it precisely: **agents buy flexibility with tokens, latency and variance.** If you can draw the shape, write the shape.

::::{dropdown} 🔍 The four patterns side by side
:color: info

```
Chaining         A -> B -> C            each step feeds the next
Routing          classify -> A | B | C  one branch runs, chosen by a classifier
Parallelization  A, B, C at once        independent subtasks, combined at the end
Orchestrator     plan -> [workers] ->   an LLM plans the subtasks, workers execute,
                 synthesize             a final step merges them
```

Anthropic's taxonomy has a fifth pattern, **evaluator-optimizer** — generate, critique, regenerate in a loop. It's important enough to get its own notebook: that's **0.5**.

All four above have a shape *you* decided in advance. That's what makes them workflows. The moment you can't draw the shape — because it depends on what the model discovers mid-task — you need the agent loop from 0.3.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Extend the chain.** Add a third step to Pattern 1 that translates the shopping list into Spanish.
2. **Add a route.** Give `route_ticket` a fourth category, `sales`, with its own specialist prompt.
3. **Time it.** Wrap Pattern 3 in `import time; t=time.time(); ... ; print(time.time()-t)`, then rewrite it as three sequential `await` calls and compare. Feel why parallelization exists.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Extend the chain:**

```python
spanish = ask("Translate to Spanish. Keep the list structure.", shopping)
print(spanish)
```

**2 — Add a route.** Two edits — and notice the type system *forces* you to make both, which is the point of the `Literal`:

```python
class Route(BaseModel):
    category: Literal["billing", "technical", "account", "sales"]

# in classify(): mention "sales" in the system prompt's JSON contract
# in route_ticket():
    elif category == "sales":
        return ask("You are a sales specialist. Answer questions about plans and upgrades.", ticket)

print(route_ticket("Can I upgrade to the annual plan and how much does it cost?"))
```

**3 — Time it:**

```python
import time

t = time.time()
await asyncio.gather(analyze("sentiment", review),
                     analyze("hardware quality", review),
                     analyze("battery", review))
parallel = time.time() - t

t = time.time()
for aspect in ["sentiment", "hardware quality", "battery"]:
    await analyze(aspect, review)
sequential = time.time() - t

print(f"parallel: {parallel:.1f}s   sequential: {sequential:.1f}s")
```

**Done when:** sequential ≈ 3× parallel (network variance aside). Independent calls should never wait in line.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValidationError` from `Route` / `Plan` | Provider without JSON mode, or the model wrapped JSON in prose | Retry, switch models, or use the `extract()` helper from 0.2 — it feeds the error back and retries |
| `SyntaxError: 'await' outside function` | Running the async cells as a plain `.py` script | Top-level `await` works in notebooks; in a script, wrap in `asyncio.run(main())` |
| **429** errors when firing parallel calls | Free-tier rate limit hit by the burst | Reduce the number of parallel calls, add small delays, or use a paid model |
| The orchestrator returns 2 or 5 subtopics instead of 3 | Models treat counts as suggestions | Validate `len(plan.subtopics)` and re-ask, or accept a range — this is what output validators (1.5) are for |
::::

**What's next:** in **0.5** we add a loop *back* into a workflow — the **reflection** pattern (generate, critique, improve) — and pause a workflow for **human approval**.